# Consolidação dos Indicadores do Ensino Médio

## 1. Configuração

In [ ]:
import polars as pl
from pathlib import Path

# Caminhos principais do projeto
ROOT = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
RAW = ROOT / "data" / "raw"
INDICADORES = ROOT / "data" / "processed" / "indicadores"
ENEM = ROOT / "data" / "processed" / "enem"

def limpar(col):
    # Converte "--" e vazios em null e ajusta decimais para Float64
    return (
        pl.col(col).cast(pl.Utf8).replace(["--", "", "NA"], None)
        .str.replace(",", ".").cast(pl.Float64, strict=False).alias(col)
    )

## 2. Indicadores do Censo Escolar (AFD, ATU, DSU e TDI)

### 2.1 AFD e Cadastro da Escola

In [ ]:
cadastro = [
    "NU_ANO_CENSO", "NO_REGIAO", "SG_UF", "CO_MUNICIPIO", "NO_MUNICIPIO",
    "CO_ENTIDADE", "NO_ENTIDADE", "NO_CATEGORIA", "NO_DEPENDENCIA",
]
afd_cols = [f"AFD_MED_CAT_{i}" for i in range(1, 6)]

# Categorias 1 a 5 de adequação da formação docente no Ensino Médio
afd = (
    pl.read_parquet(INDICADORES / "data_afd.parquet")
    .select(cadastro + [pl.col(f"MED_CAT_{i}").alias(f"AFD_MED_CAT_{i}") for i in range(1, 6)])
    .with_columns(pl.col("CO_ENTIDADE", "CO_MUNICIPIO").cast(pl.Int64, strict=False))
    .with_columns([limpar(c) for c in afd_cols])
)

### 2.2 ATU, DSU e TDI

In [ ]:
def selecionar(sigla, mapa):
    # Seleciona a chave CO_ENTIDADE e renomeia métricas do Ensino Médio
    return (
        pl.read_parquet(INDICADORES / f"data_{sigla}.parquet")
        .select([pl.col("CO_ENTIDADE").cast(pl.Int64, strict=False)]
                + [pl.col(k).alias(v) for k, v in mapa.items()])
        .with_columns([limpar(v) for v in mapa.values()])
    )

series = {"MED_CAT_0": "TOTAL", "MED_01_CAT_0": "1ANO", "MED_02_CAT_0": "2ANO", "MED_03_CAT_0": "3ANO"}

atu = selecionar("atu", {k: f"ATU_MED_{v}" for k, v in series.items()})
dsu = selecionar("dsu", {"MED_CAT_0": "DSU_MED_DOC_SUPERIOR"})
tdi = selecionar("tdi", {k: f"TDI_MED_{v}" for k, v in series.items()})

## 3. INSE e Adequação Disciplinar por UF

In [ ]:
# Preserva todas as métricas socioeconômicas e TP_CAPITAL (1=Capital, 2=Interior)
inse = (
    pl.read_parquet(INDICADORES / "data_inse.parquet")
    .select([
        pl.col("ID_ESCOLA").cast(pl.Int64, strict=False).alias("CO_ENTIDADE"),
        "NU_ANO_SAEB", "TP_TIPO_REDE", "TP_LOCALIZACAO", "TP_CAPITAL",
        "QTD_ALUNOS_INSE", "MEDIA_INSE", "INSE_CLASSIFICACAO",
        *[f"PC_NIVEL_{i}" for i in range(1, 9)],
    ])
)

# Taxa de inadequação em Física, Química e Biologia na UF (remove linha duplicada PR/PA)
adequacao_uf = (
    pl.read_parquet(INDICADORES / "data_adequacao_uf.parquet")
    .select([
        pl.col("SG_UF").str.strip_chars(),
        "TX_INADEQUACAO_FISICA", "TX_INADEQUACAO_QUIMICA", "TX_INADEQUACAO_BIOLOGIA",
    ])
    .unique(subset=["SG_UF"], keep="first", maintain_order=True)
)

## 4. Junção e Exportação

In [ ]:
# Une todas as bases pela escola (CO_ENTIDADE) e estado (SG_UF)
indicadores = (
    afd.join(atu, on="CO_ENTIDADE", how="left")
    .join(dsu, on="CO_ENTIDADE", how="left")
    .join(tdi, on="CO_ENTIDADE", how="left")
    .join(inse, on="CO_ENTIDADE", how="left")
    .join(adequacao_uf, on="SG_UF", how="left")
)

indicadores.write_parquet(INDICADORES / "indicadores_ensino_medio_2024.parquet")
print("Formato consolidado:", indicadores.shape)
indicadores.head()